# Проверка нерасчетных атрибутов по agr_id и INN

Тетрадка достает нерасчетные атрибуты напрямую из ODS/Alpha без витрины DAG.

Поля:
- cdi_id
- inn
- ogrn
- client_name_from_companies
- client_name_from_merchants
- mcc_code
- contract_number
- contract_start_dt
- contract_end_dt
- filial_rf
- vsp_name
- vsp_code
- tariff_name

Оптимизация по памяти:
- ранняя фильтрация по одному agr_id
- узкие join по ключам
- без trx-таблиц на этом шаге


In [ ]:
import pandas as pd

agr_id = "104773604399"
target_inn = "401400143905"


In [ ]:
sql = f"""
with
params as (
    select
        cast('{agr_id}' as string) as agr_id_str,
        '{target_inn}' as target_inn
),

agr_one as (
    select
        a.abs_agr_id,
        a.n_agr,
        a.c_agr_number,
        a.n_cmp_client,
        a.d_valid_from,
        a.d_valid_to
    from ods_alpha.scd1_agreements a
    join params p
      on cast(a.abs_agr_id as string) = p.agr_id_str
    where a.acq_class = 'SA'
      and a.abs_agr_id is not null
),

cmp_one as (
    select
        c.n_cmp,
        c.c_inn,
        c.c_cmp_name
    from ods_alpha.scd1_companies c
    join agr_one a
      on a.n_cmp_client = c.n_cmp
    join params p
      on c.c_inn = p.target_inn
),

r2_one as (
    select
        m.id as agr_id,
        m.c_cl_org,
        m.c_depart,
        m.c_tariff_plan
    from ods.scd1_z_r2_ip_merchants m
    join agr_one a
      on cast(m.id as string) = cast(a.abs_agr_id as string)
),

dep_one as (
    select
        d.id,
        d.c_code as vsp_code,
        d.c_name as vsp_name,
        d.c_filial
    from ods.scd1_z_depart d
    join r2_one r
      on r.c_depart = d.id
),

br_one as (
    select
        b.id,
        b.c_shortlabel as filial_rf
    from ods.scd1_z_branch b
    join dep_one d
      on d.c_filial = b.id
),

corp_one as (
    select
        z.id,
        z.c_register_gos_reg_num_rec as ogrn
    from ods.scd1_z_cl_corp z
    join r2_one r
      on r.c_cl_org = z.id
),

tariff_one as (
    select
        t.id,
        t.c_name as tariff_name
    from ods.scd1_z_r2_tariff_plan t
    join r2_one r
      on r.c_tariff_plan = t.id
),

alpha_merch_small as (
    select
        m.n_cmp,
        max(m.n_mcc) as mcc_code,
        max(m.c_mrc_name) as client_name_from_merchants
    from ods_alpha.scd1_merchants m
    join agr_one a
      on a.n_cmp_client = m.n_cmp
    group by m.n_cmp
)

select
    cast(a.abs_agr_id as string) as cdi_id,
    c.c_inn as inn,
    co.ogrn as ogrn,
    c.c_cmp_name as client_name_from_companies,
    am.client_name_from_merchants,
    am.mcc_code,
    a.c_agr_number as contract_number,
    a.d_valid_from as contract_start_dt,
    a.d_valid_to as contract_end_dt,
    b.filial_rf,
    d.vsp_name,
    d.vsp_code,
    t.tariff_name
from agr_one a
join cmp_one c on c.n_cmp = a.n_cmp_client
left join r2_one r on cast(r.agr_id as string) = cast(a.abs_agr_id as string)
left join dep_one d on d.id = r.c_depart
left join br_one b on b.id = d.c_filial
left join corp_one co on co.id = r.c_cl_org
left join tariff_one t on t.id = r.c_tariff_plan
left join alpha_merch_small am on am.n_cmp = a.n_cmp_client
;
"""

with imp:
    df_non_calc = imp.fetch(sql)

display(df_non_calc)
